In [172]:
%cd "C:\Users\Lenovo\Desktop\github projects\telcos customer churn"

C:\Users\Lenovo\Desktop\github projects\telcos customer churn


# 1.CATBOOST HYPERPARAMETER TUNING 

In [83]:
%%writefile src/model_tuner.py


import sys 
import os 
import time 
import joblib
import pandas as pd 

from dataclasses import dataclass

from sklearn.model_selection import RandomizedSearchCV,StratifiedKFold

from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report)

from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier

from src.exception import CustomException
from src.logger import logging

@dataclass
class ModelTunerConfig:


    train_data_path= os.path.join("data","final","train_selected.csv")
    test_data_path=  os.path.join("data","final","test_selected.csv")

    tuned_model_dir=os.path.join("artifacts","tuned_models")
    tuning_report_path=os.path.join("reports","hyperparameter_tuning_report.csv")


class ModelTuner:
    def __init__(self):
        
        self.config= ModelTunerConfig()

        os.makedirs(self.config.tuned_model_dir,exist_ok=True)
        os.makedirs(os.path.dirname(self.config.tuning_report_path),exist_ok=True)

        #stratifided k fold
        self.cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

    #load data
    def load_data(self):
        try:
            train_df=pd.read_csv(self.config.train_data_path)
            test_df=pd.read_csv(self.config.test_data_path)

        #split into traget and feature 
            X_train=train_df.drop(columns=["Churn"])
            y_train=train_df["Churn"]
            X_test=test_df.drop(columns=["Churn"])
            y_test=test_df["Churn"]

            logging.info(f" Training Data Shape{X_train.shape}")
            logging.info(f" Testing Data Shape{X_test.shape}")
            logging.info(f" Target Data Shape{y_train.shape}")
            logging.info(f" Target Data Shape{y_test.shape}")

            return(X_train,X_test,y_train,y_test)

        except Exception as e:
            raise CustomException(e, sys)


            #apply smote 

    def smote(self,X_train,y_train):
        try:
            logging.info("Applying SMOTE to training data")

            smote=SMOTE(random_state=42)

            X_train_smote,y_train_smote=smote.fit_resample(X_train,y_train)

            #before smote
            logging.info(f"Before applying smote{y_train.value_counts().to_dict()}")
            logging.info(f"After Applying Smote{y_train_smote.value_counts().to_dict()}")

            return(X_train_smote,y_train_smote)
        except Exception as e:
            raise CustomException(e, sys)

        #catboost hyperparameter tuning 

    def tune_catboost(self,X_train_smote,y_train_smote):
        try:
            logging.info("Catboost Hyperparameter Tuning Started")

            catboost_model=CatBoostClassifier(random_state=42,verbose=0,thread_count=2)

            param_grid={"iterations":[100,200,300],
                        "depth":[3,4,5,6,7,8],
                        "learning_rate":[0.03,0.05,0.1],
                        "l2_leaf_reg":[1,3,5],
                        "random_strength":[0,1,2]}
            
        #randomized search

            random_search=RandomizedSearchCV(estimator=catboost_model,param_distributions=param_grid,n_iter=10,
                                             scoring="roc_auc",cv=self.cv,random_state=40,n_jobs=1,verbose=10)

            start_time=time.time()
            print ("Starting Randomized Searc CV")

            random_search.fit(X_train_smote,y_train_smote)

            tune_time=time.time()-start_time

            print("Randomized Search CV Completed")

            logging.info(f"Catboost tuning completed in {tune_time:.2f} seconds")

            #best parameters
            best_params=(random_search.best_params_)
            best_cv_score=(random_search.best_score_)

            logging.info(f"Best catboostparameter{best_params}")
            logging.info(f"Best CV ROC AUC: "f"{best_cv_score:.4f}")

            return(random_search.best_estimator_,best_params,best_cv_score,tune_time)

        except Exception as e:
            raise CustomException(e, sys)


    #evaluste catboost performance 
    def evaluate_tuned_catboost(self,best_model,X_test,y_test,best_params,best_cv_score,tuning_time):
        try:
            logging.info("Evaluation Started")

            y_prob=best_model.predict_proba(X_test)[:,1]
            y_pred= (y_prob>=0.5).astype(int)

            #metrics
            accuracy = accuracy_score(y_test,y_pred)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test,y_pred,zero_division=0)
            f1 = f1_score(y_test,y_pred, zero_division=0)
            roc_auc = roc_auc_score( y_test, y_prob)



            #creating result dictionary 
            results={"Model1":"Catboost Tuned",
                     "Accuracy": round(accuracy,4),
                     "precision":round(precision,4),
                     "recall":round(recall,4),
                     "f1_score":round(f1,4),
                     "roc_auc":round(roc_auc,4),
                     "CV ROC AUC": round(best_cv_score,4),
                     "Tuning Time (Seconds)": round(tuning_time,2),
                     "Best Parameters": str(best_params)}


            #saving tuned model 

            model_path = os.path.join(self.config.tuned_model_dir,"catboost_tuned.pkl")

            joblib.dump(best_model,model_path)

            logging.info( f"Tuned CatBoost model saved to: "f"{model_path}")


          #saving result report '

            results_df=pd.DataFrame([results])
            results_df.to_csv(self.config.tuning_report_path,index=False)

            logging.info(f"CatBoost tuning report saved to: "f"{self.config.tuning_report_path}")

            return results_df

        except Exception as e:
             raise CustomException(e, sys)


Overwriting src/model_tuner.py


In [84]:
from src.model_tuner import ModelTuner

In [85]:
import importlib
import src.model_tuner as model_tuner   # your .py file

importlib.reload(model_tuner)

<module 'src.model_tuner' from 'C:\\Users\\Lenovo\\Desktop\\github projects\\telcos customer churn\\src\\model_tuner.py'>

In [86]:
tuner=ModelTuner()

In [87]:
#load data 

X_train,X_test,y_train,y_test=tuner.load_data()


print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", X_test.shape)

X_train: (5634, 45)
X_test: (1409, 45)
y_train: (5634,)
y_test: (1409, 45)


In [88]:
## apply smote 

In [89]:
X_train_smote,y_train_smote=tuner.smote(X_train,y_train)

print("X_train_smote:", X_train_smote.shape)
print("y_train_smote:", y_train_smote.shape)

X_train_smote: (8278, 45)
y_train_smote: (8278,)


In [90]:
# cat boost tuning 


In [47]:
best_catboost,best_params, best_cv_score, tuning_time= tuner.tune_catboost(X_train_smote,y_train_smote)

Starting Randomized Searc CV
Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV 1/5; 1/5] START depth=8, iterations=200, l2_leaf_reg=3, learning_rate=0.1, random_strength=2
[CV 1/5; 1/5] END depth=8, iterations=200, l2_leaf_reg=3, learning_rate=0.1, random_strength=2;, score=0.935 total time=  15.7s
[CV 2/5; 1/5] START depth=8, iterations=200, l2_leaf_reg=3, learning_rate=0.1, random_strength=2
[CV 2/5; 1/5] END depth=8, iterations=200, l2_leaf_reg=3, learning_rate=0.1, random_strength=2;, score=0.940 total time=  14.5s
[CV 3/5; 1/5] START depth=8, iterations=200, l2_leaf_reg=3, learning_rate=0.1, random_strength=2
[CV 3/5; 1/5] END depth=8, iterations=200, l2_leaf_reg=3, learning_rate=0.1, random_strength=2;, score=0.943 total time=  15.4s
[CV 4/5; 1/5] START depth=8, iterations=200, l2_leaf_reg=3, learning_rate=0.1, random_strength=2
[CV 4/5; 1/5] END depth=8, iterations=200, l2_leaf_reg=3, learning_rate=0.1, random_strength=2;, score=0.944 total time=  15.5s
[CV 5/5; 1/

In [91]:
print(best_params)
print(best_cv_score)
print(tuning_time)

{'random_strength': 2, 'learning_rate': 0.1, 'l2_leaf_reg': 3, 'iterations': 200, 'depth': 8}
0.9409165565671505
215.50323176383972


In [93]:
# evaluation 
cat_boost_results= tuner.evaluate_tuned_catboost(best_model=best_catboost,X_test=X_test,y_test=y_test,
                                                best_params=best_params,best_cv_score=best_cv_score,tuning_time=tuning_time)

cat_boost_results

,Model1,Accuracy,precision,recall,f1_score,roc_auc,CV ROC AUC,Tuning Time (Seconds),Best Parameters
0,Catboost Tuned,0.78,0.5874,0.5749,0.5811,0.8258,0.9409,215.5,"{'random_strength': 2, 'learning_rate': 0.1, '..."


# 2.ANN HYPERPARAMETER TUNING 

In [179]:
%%writefile src/ann_tuner.py

import sys 
import os
import joblib
import pandas as pd
import time
import numpy as np 
import tensorflow as tf
import keras_tuner as kt

from dataclasses import dataclass

from sklearn.model_selection import train_test_split

from sklearn.metrics import(accuracy_score,precision_score,f1_score,recall_score,roc_auc_score)

from imblearn.over_sampling import SMOTE

from src.exception import CustomException
from src.logger import logging 


@dataclass
class ANNTunerConfig:

    train_data_path="data/final/train_selected.csv"
    test_data_path= "data/final/test_selected.csv"

    tuned_model_dir="artifacts/tuned_models"
    tuned_dir= "artifacts/ann_tuner"

    tuned_report_path="reports/ann_tuning_report.csv"

class ANNTuner:
    def __init__(self):
        self.config = ANNTunerConfig()

        os.makedirs(self.config.tuned_model_dir,exist_ok=True)
        os.makedirs(self.config.tuned_dir,exist_ok=True)
        os.makedirs(os.path.dirname(self.config.tuned_report_path),exist_ok=True)


    def load_data(self):

            try:
                logging.info("Loading Final Train and test data set")

                train_df=pd.read_csv(self.config.train_data_path)
                test_df=pd.read_csv(self.config.test_data_path)

                # split x and y 
                X_train=train_df.drop(columns=["Churn"])
                y_train=train_df["Churn"]

                X_test=test_df.drop(columns=["Churn"])
                y_test=test_df["Churn"]

                logging.info(f"X_train shape{X_train.shape}")
                logging.info(f"y_train shape{X_train.shape}")
                logging.info(f"X_test shape{X_test.shape}")
                logging.info(f"y_train shape{y_test.shape}")

                return(X_train,X_test,y_train,y_test)

            except Exception as e:
                raise CustomException(e, sys)


    def apply_smote(self,X_train,y_train):

        try:
            logging.info("Initilizing SMOTE to tacke Imbalancing")

            smote=SMOTE()

            X_train_smote,y_train_smote=smote.fit_resample(X_train,y_train)

            logging.info(f"Value before SMOTE{X_train.value_counts()}")
            logging.info(f"Value before SMOTE{y_train.value_counts()}")
            logging.info(f"Value before SMOTE{X_train_smote.value_counts()}")
            logging.info(f"Value before SMOTE{y_train_smote.value_counts()}")

            return (X_train_smote,y_train_smote)

        except exception as e:
            raise CustomException(e, sys)

    def prepare_training_data(self,X_train_smote,y_train_smote):

        try:
            logging.info("Preparing training and validation set")

            X_train_final,X_val,y_train_final,y_val=train_test_split(X_train_smote,y_train_smote,test_size=0.2,random_state=42,
                                                                     stratify=y_train_smote)

            logging.info(f"X_train_final shape{X_train_final.shape}")
            logging.info(f"X_val shape{X_val.shape}")
            logging.info(f"y_train_final shape{y_train_final.shape}")
            logging.info(f"y_val_shape{y_val.shape}")

            return (X_train_final,X_val,y_train_final,y_val)
            
        except exception as e:
            raise CustomException(e, sys)


    def build_model(self,hp):

        try:
            input_dim=self.input_dim
            model=tf.keras.Sequential(name="tuned_model")
            #model define
            model.add(tf.keras.layers.Input(shape=(input_dim,)))
            #1 st dense layer 
            model.add(tf.keras.layers.Dense(hp.Int("units_1",min_value=32,max_value=256,step=32),activation="relu"))
            #dropout layer
            model.add(tf.keras.layers.Dropout(hp.Float("dropout_1",min_value=0.1,max_value=0.5,step=0.1)))
            #hidden layer 2 
            model.add(tf.keras.layers.Dense(hp.Int("units_2",min_value=16,max_value=128),activation="relu"))
            #dropout layer 2 
            model.add(tf.keras.layers.Dropout(hp.Float("dropout_2",min_value=0.1,max_value=0.5,step=0.1)))
            #output layer
            model.add(tf.keras.layers.Dense(1,activation="sigmoid"))

            # learning rate 
            learning_rate=hp.Choice("learning_rate",values=[0.01,0.001, 0.0005,0.0001])
            # model_compile 
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate)
            #model compile 
            model.compile(optimizer=optimizer,loss="binary_crossentropy",metrics=["accuracy",tf.keras.metrics.AUC(name="auc")])

            return model

        except Exception as e:
            raise CustomException(e, sys)


        #hyper tuning

    def tune_ann(self,X_train_final,y_train_final,X_val,y_val):
         try:
             logging.info(" Keras Tuner Starting")

             self.input_dim=X_train_final.shape[1]

             logging.info(f" Ann Input dim {self.input_dim}")

             tuner=kt.Hyperband(hypermodel=self.build_model,objective=kt.Objective("val_auc",direction="max"),max_epochs=25,
                                factor=3,hyperband_iterations=1,
                                directory=self.config.tuned_dir,project_name="telco_churn_ann",overwrite=True)

             ## lets add call backs 
             #Early Stopping 
             early_stopping=(tf.keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=8,restore_best_weights=True,verbose=1))

             #ReduceLearning Rate 
             reduce_lr = (tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max",factor=0.5, patience=4, min_lr=1e-6,verbose=1 ))

             # Start Timer 

             tune_start_time=time.time()

             #tuner search 
             tuner.search(X_train_final,y_train_final,validation_data=(X_val,y_val),epochs=50,batch_size=32,
                          callbacks=[early_stopping,reduce_lr],verbose=1)


             tuning_time=time.time()-tune_start_time
             logging.info("Hyperband Tuning Completed")


             #get best hyperparameter
             best_hps=(tuner.get_best_hyperparameters(num_trials=1)[0])
             logging.info("Best Hyper Parameter")

             logging.info(f"units_1: "f"{best_hps.get('units_1')}")
             logging.info(f"dropout_1: "f"{best_hps.get('dropout_1')}")
             logging.info(f"units_2: "f"{best_hps.get('units_2')}")
             logging.info(f"dropout_2: "f"{best_hps.get('dropout_2')}")
             logging.info(f"learning_rate: "f"{best_hps.get('learning_rate')}")

             #get best model
             best_model=(tuner.get_best_models(num_models=1)[0])
             best_trial=(tuner.oracle.get_best_trials(num_trials=1)[0])
             best_val_auc=best_trial.score

             return(tuner,best_model,best_hps,best_val_auc,tuning_time)

         except Exception as e :
            raise CustomException(e, sys)


    def evaluate_tuned_ann(self,best_model,X_test,y_test,best_hps,best_val_auc,tuning_time):

        try:
            logging.info(" Stating evaluation on on best model")

            y_prob=best_model.predict(X_test,verbose=0).flatten()
            y_pred= (y_prob>=0.5).astype(int)
            

            #metrics
            
            accuracy=accuracy_score(y_test,y_pred)
            
            precision=precision_score(y_test,y_pred,zero_division=0)
            
            recall=recall_score(y_test,y_pred,zero_division=0) 
            f_score=f1_score(y_test,y_pred,zero_division=0)

            roc_auc=roc_auc_score(y_test,y_prob)

            logging.info(f"Accuracy is :{accuracy}")
            logging.info(f"precision is :{precision}")
            logging.info(f"recall is :{recall}")
            logging.info(f"roc_auc_score is :{roc_auc}")


            #model saving 

            model_path = os.path.join(self.config.tuned_model_dir,"ann_tuned.keras")

            best_model.save(model_path)

            logging.info("model saved to {model_path}")

            # creating results 
            results = {"Model": "ANN Tuned", "Accuracy": accuracy,"Precision": precision,"Recall": recall,
                        "F1 Score":f_score,"ROC AUC": roc_auc,"Best Validation ROC AUC":best_val_auc,"Tuning Time":tuning_time,
                        "units_1": best_hps.get("units_1"),"units_2":best_hps.get("units_2"),"dropout_1":best_hps.get("dropout_1"),
                        "dropout_2":best_hps.get("dropout_2"),"learning_rate":best_hps.get("learning_rate")}


            results_df=pd.DataFrame([results])

            results_df.to_csv( self.config.tuned_report_path,index=False)
            logging.info(f"Results_df save to path:{self.config.tuned_report_path}")
            return results_df
        except Exception as e:
             raise CustomException(e, sys)


            

           






             
             


           
         
        
           
        
     



                      
            


            

            
        

Overwriting src/ann_tuner.py


In [180]:
import importlib
import src.ann_tuner as ann_tuner   # your .py file

importlib.reload(ann_tuner)

<module 'src.ann_tuner' from 'C:\\Users\\Lenovo\\Desktop\\github projects\\telcos customer churn\\src\\ann_tuner.py'>

In [181]:
from src.ann_tuner import ANNTuner,ANNTunerConfig

In [182]:
config=ANNTunerConfig()

In [183]:
ann_tuner=ANNTuner()

In [134]:
## load data 

X_train,X_test,y_train,y_test=ann_tuner.load_data()

In [135]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5634, 45)
(1409, 45)
(5634,)
(1409,)


In [136]:
print(y_train.value_counts())

Churn
0    4139
1    1495
Name: count, dtype: int64


In [137]:
## apply smote 

In [138]:
X_train_smote,y_train_smote=ann_tuner.apply_smote(X_train,y_train)

In [139]:
print(X_train_smote.shape)
print(y_train_smote.shape)

(8278, 45)
(8278,)


In [140]:
# preparint training data 

X_train_final,X_val,y_train_final,y_val=ann_tuner.prepare_training_data(X_train_smote,y_train_smote)

In [141]:
print("X_train_final:", X_train_final.shape)
print("X_val        :", X_val.shape)

print("y_train_final:", y_train_final.shape)
print("y_val        :", y_val.shape)

X_train_final: (6622, 45)
X_val        : (1656, 45)
y_train_final: (6622,)
y_val        : (1656,)


In [143]:
#starting tuning 

tuner,best_model,best_hps,best_val_auc,tuning_time=ann_tuner.tune_ann(X_train_final=X_train_final
                                                                      ,y_train_final=y_train_final,X_val=X_val,y_val=y_val)

Trial 30 Complete [00h 00m 20s]
val_auc: 0.899242103099823

Best val_auc So Far: 0.9081505537033081
Total elapsed time: 00h 04m 37s


C:\Users\Lenovo\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [148]:
print(best_hps.get('units_1'))
print(best_hps.get('units_2'))
print(best_hps.get('dropout_1'))
print(best_hps.get('dropout_2'))
print(best_hps.get('learning_rate'))


192
50
0.30000000000000004
0.1
0.001


In [151]:
best_model.summary()

Model: "tuned_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 192)            │         8,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 192)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │         9,650 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,533 (72.39 KB)

 Trainable params: 18,533 (72.39 KB)

 Non-trainable params: 0 (0.00 B)

In [150]:
print(best_hps)

In [184]:
## ann_tuned results 

evaluate_tuned_result=(ann_tuner.evaluate_tuned_ann(best_model=best_model,X_test=X_test,y_test=y_test,best_hps=best_hps,
                                                  best_val_auc=best_val_auc,tuning_time=tuning_time))

In [185]:
evaluate_tuned_result

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,Best Validation ROC AUC,Tuning Time,units_1,units_2,dropout_1,dropout_2,learning_rate
0,ANN Tuned,0.773598,0.558887,0.697861,0.62069,0.823088,0.908151,276.951706,192,50,0.3,0.1,0.001
